# Session 5 — Baseline for the Nim dataset

This notebook follows Session 5's supervised-learning workflow: define `X` and `y`, use a stratified train/test split, fit a simple baseline, and evaluate it with accuracy, precision, recall, and a confusion matrix.

The dataset is all 3-pile Nim positions with pile sizes from 0 to 7. The target `loses=True` means the player whose turn it is is in a losing position under perfect play.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, classification_report,
    ConfusionMatrixDisplay
)

## 1. Generate the labeled dataset

The Nim losing-position rule is that the XOR (`nim_sum`) of the pile sizes is zero. We enumerate all 512 positions and use that rule to create labels.

In [ ]:
rows = []
for p1 in range(8):
    for p2 in range(8):
        for p3 in range(8):
            nim_sum = p1 ^ p2 ^ p3
            rows.append({
                "p1": p1, "p2": p2, "p3": p3,
                "nim_sum": nim_sum,
                "loses": nim_sum == 0
            })

df = pd.DataFrame(rows)
print("Shape:", df.shape)
display(df.head())
print("\nTarget distribution:")
print(df["loses"].value_counts())
print("\nTarget proportion:")
print(df["loses"].value_counts(normalize=True).round(3))

## 2. Define features and target

For this baseline, use the raw pile sizes only—do **not** include `nim_sum`, because it directly encodes the rule that determines the label and would leak the answer into the features.

In [ ]:
X = df[["p1", "p2", "p3"]]
y = df["loses"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=0, stratify=y
)
print("Train rows:", len(X_train))
print("Test rows:", len(X_test))

## 3. Baseline model

`DummyClassifier(strategy="most_frequent")` ignores the pile values and always predicts the majority class. This is the reference point that a learned model should beat.

In [ ]:
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)
baseline_pred = baseline.predict(X_test)

print("Baseline predicted class:", baseline.classes_[np.argmax(baseline.class_prior_)])
print("Accuracy:", round(accuracy_score(y_test, baseline_pred), 3))
print("Precision (loses):", round(precision_score(y_test, baseline_pred, zero_division=0), 3))
print("Recall (loses):", round(recall_score(y_test, baseline_pred, zero_division=0), 3))

## 4. Evaluation report and confusion matrix

In [ ]:
print(classification_report(
    y_test, baseline_pred,
    target_names=["wins (False)", "loses (True)"],
    zero_division=0
))

ConfusionMatrixDisplay.from_predictions(
    y_test, baseline_pred,
    labels=[False, True],
    display_labels=["wins", "loses"]
)
plt.title("Session 5 Baseline — Confusion Matrix")
plt.tight_layout()
plt.show()

## 5. Baseline interpretation

- The majority-class baseline predicts **wins** for every position.
- The full dataset contains 448 winning positions and 64 losing positions (87.5% / 12.5%).
- A stratified 70/30 split preserves this class imbalance approximately in both sets.
- Expected baseline accuracy is approximately **87.5%**, despite the model learning nothing about Nim.
- Precision and recall for `loses=True` are **0**, because the baseline never predicts the minority class.

**Conclusion:** accuracy alone is misleading for this dataset. Any model comparison should report minority-class recall and the confusion matrix, and should demonstrate improvement over this baseline.